In [25]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from dig4bio.io import read_raman_file
from sklearn.linear_model import LinearRegression
import numpy as np
from dig4bio.constants import FINGERPRINT_GRID

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
df = read_raman_file('source_datasets.csv', 'processed', subfolder='source_grid_fingerprint_linear')

In [27]:
source_devices_all = df['device'].unique().tolist()
fold_idxs_all = df['fold_idx'].unique().tolist()

"""
for target_device in devices:
    for k in 0,1,2,3,4:
        target_test = target_device where fold_idx == k

        target_calib = target_device where fold_idx != k

        source_train = other devices where fold_idx != k

        train/adapt on source_train + target_calib
        predict target_test 

"""

def generate_transfer_dataframes(df, test_device: str, fold_idx: int):

    source_df = df[df['device'] != test_device]
    transfer_test_df = df[df['device'] == test_device]

    source_train = source_df[source_df['fold_idx'] != fold_idx]
    target_calibration = transfer_test_df[transfer_test_df['fold_idx'] != fold_idx]
    target_test = transfer_test_df[transfer_test_df['fold_idx'] == fold_idx]
    
    return source_train, target_calibration, target_test


for test_device in source_devices_all[:1]:
    for fold_idx in fold_idxs_all[:1]:

        source_devices = source_devices_all.copy()
        source_devices.remove(test_device)

        source_train,target_calibration, target_test  = generate_transfer_dataframes(df, test_device, fold_idx)

        display(target_test.sample(4,random_state=42))
        display(target_calibration.sample(4,random_state=42))
        display(source_train.sample(5,random_state=43))

        print(source_devices)

,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
13,2976.44,2970.110,2963.78,2963.535,2963.29,2954.535,2945.78,2922.025,2898.27,2883.940,...,1160.705,1160.94,1143.465,1125.99,0.51645,1.02701,1.32862,0.0,0,anton532
39,2902.52,2900.100,2897.68,2892.695,2887.71,2871.710,2855.71,2833.605,2811.50,2799.405,...,1105.520,1109.04,1097.355,1085.67,0.26229,0.03709,1.49460,0.0,0,anton532
30,2990.98,2989.425,2987.87,2985.405,2982.94,2967.505,2952.07,2924.950,2897.83,2881.650,...,1176.050,1176.45,1163.010,1149.57,0.00000,0.76773,3.94541,0.0,0,anton532
45,3149.59,3151.150,3152.71,3149.690,3146.67,3128.230,3109.79,3082.930,3056.07,3042.270,...,1209.255,1209.90,1195.285,1180.67,0.00000,0.00000,0.00000,0.0,0,anton532


,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
182,3032.87,3038.015,3043.16,3043.245,3043.33,3026.485,3009.64,2982.125,2954.61,2936.360,...,1212.280,1215.73,1204.510,1193.29,10.16120,0.00000,1.477840,0.0,3,anton532
198,3477.73,3481.160,3484.59,3488.040,3491.49,3478.990,3466.49,3439.505,3412.52,3392.825,...,1965.765,1968.48,1955.340,1942.20,16.73440,0.00000,0.022216,1.0,3,anton532
143,3270.11,3276.875,3283.64,3283.625,3283.61,3262.625,3241.64,3208.085,3174.53,3159.360,...,1499.995,1501.37,1484.965,1468.56,6.39311,0.00000,0.024451,1.0,2,anton532
230,3057.72,3057.830,3057.94,3052.780,3047.62,3027.240,3006.86,2978.870,2950.88,2936.785,...,1241.680,1242.98,1227.875,1212.77,4.62097,0.97254,0.014210,0.0,4,anton532


,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
974,4833.824034,4843.491379,4785.732759,4727.974138,4732.643777,4751.957082,4771.517241,4791.344828,4803.536481,4775.639485,...,1393.181287,1385.341176,1350.635294,1358.052632,2.03291,1.50492,0.009459,0.0,3,metrohm
2234,36894.033899,36710.827120,36553.963579,36390.218691,36213.187403,36027.180624,35859.656112,35752.821221,35694.337527,35596.434224,...,3881.329578,3944.889073,3907.349477,3803.478325,1.98235,0.73691,0.004251,0.0,4,tornado
1561,2355.787238,2375.052369,2337.815112,2284.021273,2220.591156,2081.085505,2087.792818,2155.467938,2278.292360,2556.722353,...,2561.782756,2567.322317,2556.872461,2550.006859,0.25649,0.74089,3.345560,0.0,2,tec
1331,6109.329436,6063.788622,6026.694006,6008.803536,6018.246737,6052.043536,6078.373405,6070.104774,6046.683559,6039.791078,...,879.091018,860.677791,872.386873,875.236224,0.56078,1.07925,3.139890,0.0,4,mettler
796,4375.703863,4389.068966,4351.137931,4313.206897,4305.304721,4304.446352,4309.379310,4320.586207,4325.055794,4294.154506,...,1353.046784,1357.282353,1357.870588,1353.894737,0.79631,1.06674,3.090570,0.0,1,metrohm


['anton785', 'kaiser', 'metrohm', 'mettler', 'tec', 'timegate', 'tornado']


In [28]:
def train_model(model, train_df: pd.DataFrame, feature_columns: list[str], label_columns: list[str]) -> LinearRegression:

    x_train = train_df[feature_columns]
    y_train = train_df[label_columns]

    model.fit(x_train,y_train)

    return model

def calculate_model_error(true_values, predictions, error_method):

    if error_method == 'residual':
        errors = true_values - predictions
    else:
        errors = 0

    return errors

def calibrate_model(prediction_model,calibration_model_type, calibration_df: pd.DataFrame, wavenumber_columns: list[str], label_columns: list[str]) -> LinearRegression:

    calibration_model = calibration_model_type()

    x_calibrate = calibration_df[wavenumber_columns]
    y_calibrate = calibration_df[label_columns].to_numpy()

    predictions = prediction_model.predict(x_calibrate)

    errors = calculate_model_error(y_calibrate, predictions, error_method = 'residual')

    calibration_model.fit(x_calibrate,errors)

    return calibration_model

def measure_model_performance(test_df: pd.DataFrame, prediction_model, calibration_model, wavenumber_columns: list[str], label_columns: list[str]) -> pd.DataFrame:

    x_test = test_df[wavenumber_columns]
    y_test = test_df[label_columns]

    initial_predictions = prediction_model.predict(x_test)
    calibrated_predictions = initial_predictions + calibration_model.predict(x_test)
    initial_errors = y_test - initial_predictions
    calibrated_errors = y_test - calibrated_predictions

    combined_data = np.concatenate([y_test,initial_predictions,calibrated_predictions,initial_errors,calibrated_errors],axis=1)

    combined_columns = (
        [i + '_true' for i in label_columns] +
        [i + '_predicted' for i in label_columns] +
        [i + '_calibrated' for i in label_columns] + 
        [i + '_predicted_error' for i in label_columns] +
        [i + '_calibrated_error' for i in label_columns]                        
    )

    model_performance = pd.DataFrame(combined_data, columns = combined_columns)

    return model_performance

test_device = source_devices_all[0]
fold_idx = 0

wavenumber_columns = FINGERPRINT_GRID.astype(str)
label_columns = ['glucose','Na_acetate']

source_devices = source_devices_all.copy()
source_devices.remove(test_device)

model = LinearRegression()

source_train, target_calibration, target_test  = generate_transfer_dataframes(df, test_device, fold_idx)
prediction_model = train_model(model, source_train, wavenumber_columns,label_columns)
calibration_model = calibrate_model(prediction_model, LinearRegression, target_calibration, wavenumber_columns, label_columns)

model_performance = measure_model_performance(target_test, prediction_model, calibration_model,wavenumber_columns, label_columns=['glucose','Na_acetate'])


In [29]:
model_performance.mean()

glucose_true                    2.138600
Na_acetate_true                 0.468878
glucose_predicted             -17.309951
Na_acetate_predicted           -0.427591
glucose_calibrated              3.221808
Na_acetate_calibrated           0.422477
glucose_predicted_error        19.448551
Na_acetate_predicted_error      0.896469
glucose_calibrated_error       -1.083208
Na_acetate_calibrated_error     0.046401
dtype: float64

In [30]:
device_errors={}

for test_device in source_devices_all:

    device_glucose_errors = []
    device_na_acetate_errors = []

    for fold_idx in fold_idxs_all:

        source_devices = source_devices_all.copy()
        source_devices.remove(test_device)

        source_train,target_calibration, target_test  = generate_transfer_dataframes(df, test_device, fold_idx)

        model = LinearRegression()
        
        prediction_model = train_model(model, source_train, wavenumber_columns, label_columns)
        calibration_model = calibrate_model(prediction_model, LinearRegression, target_calibration, wavenumber_columns,label_columns)
        
        model_performance = measure_model_performance(target_test, prediction_model, calibration_model,wavenumber_columns,['glucose','Na_acetate'])
        errors = model_performance[['glucose_calibrated_error','Na_acetate_calibrated_error']].mean().tolist()

        device_glucose_errors.append(errors[0])
        device_na_acetate_errors.append(errors[1])
    
    device_errors[test_device] = {'glucose': np.mean(device_glucose_errors).round(3), 'na_acetate': np.mean(device_na_acetate_errors).round(3)}

print(device_errors)


{'anton532': {'glucose': np.float64(-0.412), 'na_acetate': np.float64(-0.035)}, 'anton785': {'glucose': np.float64(-0.291), 'na_acetate': np.float64(0.008)}, 'kaiser': {'glucose': np.float64(0.178), 'na_acetate': np.float64(-0.03)}, 'metrohm': {'glucose': np.float64(-1.825), 'na_acetate': np.float64(-0.176)}, 'mettler': {'glucose': np.float64(2.68), 'na_acetate': np.float64(-0.184)}, 'tec': {'glucose': np.float64(-0.481), 'na_acetate': np.float64(0.205)}, 'timegate': {'glucose': np.float64(-0.04), 'na_acetate': np.float64(-0.022)}, 'tornado': {'glucose': np.float64(13.568), 'na_acetate': np.float64(1.855)}}


In [31]:
pd.DataFrame(device_errors)

,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.412,-0.291,0.178,-1.825,2.680,-0.481,-0.040,13.568
na_acetate,-0.035,0.008,-0.030,-0.176,-0.184,0.205,-0.022,1.855


In [32]:
def augmented_cross_validate(
        source_df: pd.DataFrame,
        devices: list[str],
        fold_indices: list[str],
        wavenumber_columns: list[str],
        label_columns: list[str],
        prediction_model_type = LinearRegression,
        calibration_model_type = LinearRegression
    ) -> dict:
    
    device_errors={}
    for device in devices:
        device_fold_errors = {}

        for fold_idx in fold_indices:

            source_devices = devices.copy()
            source_devices.remove(device)

            source_train, target_calibration, target_test  = generate_transfer_dataframes(source_df, device, fold_idx)

            model = prediction_model_type()
            
            prediction_model = train_model(model, source_train, wavenumber_columns, label_columns)
            calibration_model = calibrate_model(prediction_model, calibration_model_type, target_calibration, wavenumber_columns, label_columns)
            
            model_performance = measure_model_performance(target_test, prediction_model, calibration_model,wavenumber_columns,label_columns)
            errors = model_performance[[i + '_calibrated_error' for i in label_columns]].mean().tolist()

            device_fold_errors[fold_idx]={label: error for label,error in zip(label_columns, errors)}
        
    
        device_errors[device] = device_fold_errors

    return device_errors


stats = augmented_cross_validate(
    source_df=df,
    devices = source_devices_all,
    fold_indices=fold_idxs_all,
    wavenumber_columns = FINGERPRINT_GRID.astype(str),
    label_columns=['glucose','Na_acetate']
)
display(stats)

{'anton532': {0: {'glucose': -1.0832075202484228,
   'Na_acetate': 0.04640144442940761},
  1: {'glucose': -1.3202031744413685, 'Na_acetate': -0.053147138270467534},
  2: {'glucose': -0.0741688868033393, 'Na_acetate': 0.22739051859868073},
  3: {'glucose': -0.4050683294929692, 'Na_acetate': -0.16061681001200295},
  4: {'glucose': 0.8245527827191028, 'Na_acetate': -0.23416707087113012}},
 'anton785': {0: {'glucose': 0.603452099868387,
   'Na_acetate': -0.1033942328566773},
  1: {'glucose': -1.844651909042955, 'Na_acetate': 0.08922231513360032},
  2: {'glucose': 0.25680105499198713, 'Na_acetate': -0.0008118347939028184},
  3: {'glucose': 0.3219719579965817, 'Na_acetate': 0.016706129686252437},
  4: {'glucose': -0.7934153012318736, 'Na_acetate': 0.03593598040910069}},
 'kaiser': {0: {'glucose': -0.6152640447841597,
   'Na_acetate': 0.06726163829310354},
  1: {'glucose': 0.7608940530734286, 'Na_acetate': -0.060608866153633784},
  2: {'glucose': 0.1443138722309925, 'Na_acetate': -0.132012713

In [33]:
for idx,val in stats.items():
    display(pd.DataFrame(val))
    break

,0,1,2,3,4
glucose,-1.083208,-1.320203,-0.074169,-0.405068,0.824553
Na_acetate,0.046401,-0.053147,0.227391,-0.160617,-0.234167


In [34]:
from sklearn.metrics import r2_score
import numpy as np

print(np.array(r2_score([[1,2,3],[7,8,9]],[[5,2,6],[2,6,8]],multioutput='raw_values')))
print(np.array(r2_score([1,2],[1,2])))

# How should we calculate the r^2 when doing cross validation? We could calculate the r^2 per fold and then average them? Or we could combine all the out-of-fold predictions
# into one dataset and calculate the r^2 on that? But then there may be size imbalances.

source_df = read_raman_file('source_datasets',level='processed',subfolder='source_grid_fingerprint_linear')

display(source_df['fold_idx'].value_counts().sort_index())

# It looks like the folds are equally balanced. That's great news for combining the fold predictions then taking an average.

[-1.27777778  0.77777778  0.44444444]
1.0


fold_idx
0    437
1    454
2    458
3    454
4    458
Name: count, dtype: int64

In [35]:
mask = source_df['fold_idx']==1

source_df.loc[mask]

,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
50,3519.360000,3519.375000,3519.390000,3518.345000,3517.300000,3501.590000,3485.880000,3457.775000,3429.670000,3408.480000,...,1666.865000,1667.850000,1654.145000,1640.440000,10.74480,1.05836,0.005669,0.0,1,anton532
51,3420.500000,3415.660000,3410.820000,3407.410000,3404.000000,3388.490000,3372.980000,3344.960000,3316.940000,3300.380000,...,1511.715000,1512.700000,1498.690000,1484.680000,10.74480,1.05836,0.005669,0.0,1,anton532
52,3402.650000,3396.740000,3390.830000,3388.195000,3385.560000,3370.295000,3355.030000,3322.300000,3289.570000,3269.330000,...,1482.510000,1484.680000,1470.350000,1456.020000,10.74480,1.05836,0.005669,0.0,1,anton532
53,3377.720000,3384.820000,3391.920000,3386.240000,3380.560000,3356.060000,3331.560000,3307.920000,3284.280000,3267.385000,...,1453.065000,1455.870000,1442.185000,1428.500000,10.74480,1.05836,0.005669,0.0,1,anton532
54,3367.810000,3365.230000,3362.650000,3355.675000,3348.700000,3330.435000,3312.170000,3291.295000,3270.420000,3255.760000,...,1444.865000,1447.140000,1432.605000,1418.070000,10.74480,1.05836,0.005669,0.0,1,anton532
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021,39671.901111,39496.344617,39393.892910,39363.104392,39305.922543,39117.758099,38873.732894,38744.231382,38765.889179,38825.299562,...,5221.493714,5192.862775,5170.682282,5152.397616,1.02342,0.71756,1.557450,0.0,1,tornado
2022,39889.231592,39677.100911,39451.031939,39302.145951,39186.486500,39034.553222,38871.755946,38774.187395,38754.584118,38754.392866,...,5225.698119,5229.845308,5192.558679,5136.991162,1.02342,0.71756,1.557450,0.0,1,tornado
2023,40089.326624,39872.186847,39654.282412,39442.627539,39283.020760,39220.006630,39188.496744,39067.080594,38869.057675,38758.195548,...,5259.499825,5213.884462,5163.739785,5144.039494,1.02342,0.71756,1.557450,0.0,1,tornado
2024,39848.411092,39603.122182,39386.253259,39267.021053,39206.526894,39141.032200,39040.611436,38895.295761,38726.122601,38588.367310,...,5220.473955,5212.363505,5155.383337,5096.971516,1.02342,0.71756,1.557450,0.0,1,tornado


In [ ]:
# Let's attempt the pipeline now we've implemented everything properly
%load_ext autoreload
%autoreload 2
from dig4bio.io import read_raman_file
from dig4bio.cv import cross_validate_sample_folds
from dig4bio.constants import FINGERPRINT_GRID, LABEL_COLUMNS
from dig4bio.models import create_model_factory
from sklearn.linear_model import LinearRegression

source_devices_df = read_raman_file('source_datasets',level='processed',subfolder='source_grid_fingerprint_linear')

source_devices_all = source_devices_df['device'].unique().tolist()
fold_idxs_all = source_devices_df['fold_idx'].unique().tolist()

model_factory = create_model_factory(
    prediction_model= LinearRegression(),
    calibration_model= LinearRegression()
)

device_scores = cross_validate_sample_folds(
    df = source_devices_df,
    wavenumber_columns = FINGERPRINT_GRID.astype(str),
    label_columns = LABEL_COLUMNS,
    model_factory = model_factory,
    calibrate=True
)

display(pd.DataFrame(device_scores).T)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,glucose,Na_acetate,Mg_SO4
anton532,0.250473,-0.690651,0.501794
anton785,0.134544,0.209264,0.487199
kaiser,0.535335,0.728879,0.505885
metrohm,-77.466291,-22.011849,-17.844472
mettler,-83.640601,-85.078768,-34.417750
tec,-11.096872,-15.242137,-1.706664
timegate,0.517171,0.690905,0.694120
tornado,-2109.214342,-714.159917,-1027.629457


In [ ]:
%load_ext autoreload
%autoreload 2
from dig4bio.cv import cross_validate_sample_folds
from dig4bio.constants import FINGERPRINT_GRID, LABEL_COLUMNS
from dig4bio.models import create_model_factory
from sklearn.linear_model import LinearRegression

model_factory = create_model_factory(
    prediction_model= LinearRegression()
)

device_scores = cross_validate_sample_folds(
    df = source_devices_df,
    wavenumber_columns = FINGERPRINT_GRID.astype(str),
    label_columns = LABEL_COLUMNS,
    model_factory = model_factory,
    calibrate = False
)

display(pd.DataFrame(device_scores).T)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,glucose,Na_acetate,Mg_SO4
anton532,-0.578226,-0.388733,0.025531
anton785,-0.572072,-1.113410,-0.081151
kaiser,-0.650572,-0.187411,0.576659
metrohm,-1.436235,-0.112135,0.412636
mettler,-1.043655,-0.163476,0.398323
tec,-1.263017,-0.256038,0.541820
timegate,-0.932435,-0.940740,-0.255615
tornado,-0.380245,-0.433969,0.575777
